<a href="https://colab.research.google.com/github/ReethuS10/Neurodivergence/blob/main/AImodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers torch torchvision pillow gradio accelerate sentencepiece

In [3]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import gradio as gr

In [4]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

print("✅ BLIP model loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

✅ BLIP model loaded successfully!


In [7]:
from PIL import Image
from google.colab import files

uploaded = files.upload()

image_path = next(iter(uploaded))
image = Image.open(image_path).convert("RGB")

inputs = processor(image, return_tensors="pt")
out = model.generate(**inputs)

description = processor.decode(out[0], skip_special_tokens=True)

# Simple category detection
description_lower = description.lower()

if any(word in description_lower for word in ["shoe", "sneaker", "nike", "adidas", "boot", "air max"]):
    category = "Footwear"

elif any(word in description_lower for word in ["shirt", "t-shirt", "hoodie", "dress", "jeans"]):
    category = "Clothing"

elif any(word in description_lower for word in ["phone", "iphone", "mobile", "smartphone"]):
    category = "Electronics"

elif any(word in description_lower for word in ["laptop", "computer", "notebook"]):
    category = "Computers"

else:
    category = "Other"

print("\n" + "="*55)
print("        AI PRODUCT UNDERSTANDING SYSTEM")
print("="*55)

print(f"📦 Product Category    : {category}")
print(f"📝 Product Description : {description}")
print(f"🤖 AI Model            : BLIP + Rule-based Classifier")
print(f"🖼️ Input Image         : {image_path}")

print("="*55)

Saving OPAC.jpg to OPAC (2).jpg

        AI PRODUCT UNDERSTANDING SYSTEM
📦 Product Category    : Footwear
📝 Product Description : nike air max ultra
🤖 AI Model            : BLIP + Rule-based Classifier
🖼️ Input Image         : OPAC (2).jpg


In [8]:
!pip -q install ftfy regex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00


In [9]:
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("✅ CLIP model loaded successfully!")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ CLIP model loaded successfully!


In [10]:
from PIL import Image

# Candidate categories
labels = [
    "Footwear",
    "Clothing",
    "Electronics",
    "Furniture",
    "Sports Equipment",
    "Accessories"
]

image = Image.open(image_path).convert("RGB")

inputs = clip_processor(
    text=labels,
    images=image,
    return_tensors="pt",
    padding=True
)

outputs = clip_model(**inputs)

logits = outputs.logits_per_image
probs = logits.softmax(dim=1)

best_index = probs.argmax().item()

print("Predicted Category :", labels[best_index])
print("Confidence :", round(probs[0][best_index].item()*100,2), "%")

Predicted Category : Footwear
Confidence : 87.45 %


In [11]:
def analyze_product(image_path):
    from PIL import Image

    image = Image.open(image_path).convert("RGB")

    # ---------- BLIP ----------
    inputs = processor(image, return_tensors="pt")
    out = model.generate(**inputs)
    description = processor.decode(out[0], skip_special_tokens=True)

    # ---------- CLIP ----------
    labels = [
        "Footwear",
        "Clothing",
        "Electronics",
        "Furniture",
        "Sports Equipment",
        "Accessories"
    ]

    clip_inputs = clip_processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    )

    outputs = clip_model(**clip_inputs)

    probs = outputs.logits_per_image.softmax(dim=1)

    index = probs.argmax().item()

    category = labels[index]
    confidence = probs[0][index].item() * 100

    print("="*60)
    print("        AI PRODUCT UNDERSTANDING SYSTEM")
    print("="*60)
    print("📦 Category      :", category)
    print("📝 Description   :", description)
    print("📊 Confidence    :", f"{confidence:.2f}%")
    print("="*60)

In [12]:
analyze_product(image_path)

        AI PRODUCT UNDERSTANDING SYSTEM
📦 Category      : Footwear
📝 Description   : nike air max ultra
📊 Confidence    : 87.45%


In [13]:
def analyze(image):

    # ---------- BLIP ----------
    inputs = processor(image, return_tensors="pt")
    out = model.generate(**inputs)
    description = processor.decode(out[0], skip_special_tokens=True)

    # ---------- CLIP ----------
    labels = [
        "Footwear",
        "Clothing",
        "Electronics",
        "Furniture",
        "Sports Equipment",
        "Accessories"
    ]

    clip_inputs = clip_processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    )

    outputs = clip_model(**clip_inputs)

    probs = outputs.logits_per_image.softmax(dim=1)

    index = probs.argmax().item()

    category = labels[index]
    confidence = round(probs[0][index].item() * 100, 2)

    return category, description, f"{confidence}%"

In [15]:
!pip -q install transformers torch torchvision pillow gradio accelerate sentencepiece ftfy regex

In [16]:
import torch
import gradio as gr
from PIL import Image

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    CLIPProcessor,
    CLIPModel
)

In [17]:
print("Loading BLIP model...")

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

print("Loading CLIP model...")

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("✅ Models Loaded Successfully!")

Loading BLIP model...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading CLIP model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ Models Loaded Successfully!


In [18]:
labels = [
    "Footwear",
    "Clothing",
    "Electronics",
    "Furniture",
    "Sports Equipment",
    "Accessories",
    "Books",
    "Beauty Products",
    "Kitchen Appliances",
    "Watches",
    "Bags",
    "Toys",
    "Home Decor"
]

In [19]:
def analyze(image):

    # ---------------- BLIP ----------------

    inputs = processor(image, return_tensors="pt")

    output = blip_model.generate(
        **inputs,
        max_new_tokens=40
    )

    description = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    # ---------------- CLIP ----------------

    clip_inputs = clip_processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    )

    outputs = clip_model(**clip_inputs)

    probs = outputs.logits_per_image.softmax(dim=1)

    best = probs.argmax().item()

    category = labels[best]

    confidence = round(
        probs[0][best].item() * 100,
        2
    )

    return (
        image,
        category,
        description,
        f"{confidence}%"
    )

In [ ]:
demo = gr.Interface(
    fn=analyze,

    inputs=gr.Image(
        type="pil",
        label="📤 Upload Product Image"
    ),

    outputs=[
        gr.Image(label="🖼 Uploaded Image"),
        gr.Textbox(label="📦 Product Category"),
        gr.Textbox(label="📝 Product Description"),
        gr.Textbox(label="📊 Confidence Score")
    ],

    title="Multimodal AI Product Understanding System",

    description="""
Upload a product image.

The system will:

✅ Generate Product Description (BLIP)

✅ Predict Product Category (CLIP)

✅ Display Confidence Score
    """,

    theme="soft"
)

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6a31f590c4753eb3d0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
